In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import re
import unicodedata
import plotly.graph_objects as go
import geopandas as gpd
from shapely.geometry import Point
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_seq_items", None)

## Direcciones

os.getcwd()
os.chdir("C:\\Users\\Usuario\\OneDrive - Global Green Growth Institute\\Documentos\\2025\\Outputs\\Output1\\Stress Test\\3.Data\\")
print("Lista de archivos:", os.listdir())

## Carga de precipitación diaria: funciones auxiliares

In [ ]:
STATION_COLS = [
    "CodigoEstacion",
    "NombreEstacion",
    "Departamento",
    "Municipio",
    "ZonaHidrografica",
    "Latitud",
    "Longitud",
]

RENAME_COLUMNAS = {
    "codigoestacion": "CodigoEstacion",
    "nombreestacion": "NombreEstacion",
    "departamento": "Departamento",
    "municipio": "Municipio",
    "zonahidrografica": "ZonaHidrografica",
    "latitud": "Latitud",
    "longitud": "Longitud",
    "valorobservado": "ValorObservado",
    "fechaobservacion": "FechaObservacion",
}


def limpiar_decimales(df, columnas=("Latitud", "Longitud", "ValorObservado")):
    """Convierte columnas numericas que pueden venir con coma decimal a float."""
    for col in columnas:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.strip()
            .replace("nan", None)
            .astype(float)
        )
    return df


def calcular_diaria(df):
    """Agrega observaciones sub-diarias a nivel estacion-fecha."""
    return (
        df.groupby(STATION_COLS + ["fecha"], as_index=False)
        .agg(
            precip_min_10min=("ValorObservado", "min"),
            precip_max_10min=("ValorObservado", "max"),
            precip_media_10min=("ValorObservado", "mean"),
            precip_acum_diaria=("ValorObservado", "sum"),
        )
        .sort_values(["CodigoEstacion", "fecha"])
    )

### Fuente 1: Precipitación_20251222.csv

In [ ]:
precipitacion = pd.read_csv("Precipitación_20251222.csv")
precipitacion = limpiar_decimales(precipitacion)
precipitacion["fecha"] = pd.to_datetime(
    precipitacion["FechaObservacion"].str.slice(0, 11),
    format="%Y %b %d",
    errors="coerce",
    cache=True,
)

df_daily = calcular_diaria(precipitacion)
df_daily.head()

### Fuente 2: Precipitación_20251222_2.csv

In [ ]:
data_2 = pd.read_csv("Precipitación_20251222_2.csv")
data_2 = limpiar_decimales(data_2)
data_2["fecha"] = pd.to_datetime(
    data_2["FechaObservacion"].str.slice(0, 11),
    format="%Y %b %d",
    errors="coerce",
    cache=True,
)

df_daily_2 = calcular_diaria(data_2)
df_daily_2.head()

### Fuente 3: Precipitación_2022.csv

In [ ]:
precipitacion_2022 = pd.read_csv("Precipitación_2022.csv")
precipitacion_2022 = precipitacion_2022.rename(columns=RENAME_COLUMNAS)
precipitacion_2022["fecha"] = pd.to_datetime(
    precipitacion_2022["FechaObservacion"].str.slice(0, 10),
    errors="coerce",
    cache=True,
)

df_daily_3 = calcular_diaria(precipitacion_2022)
df_daily_3.head()

### Fuentes 4 y 5: precipitacion_2021_1.csv y precipitacion_2021_2.csv

In [ ]:
precipitacion_2021 = pd.concat(
    [
        pd.read_csv("precipitacion_2021_1.csv"),
        pd.read_csv("precipitacion_2021_2.csv"),
    ],
    ignore_index=True,
).rename(columns=RENAME_COLUMNAS)

precipitacion_2021["fecha"] = pd.to_datetime(precipitacion_2021["fecha"], errors="coerce")

df_daily_4 = calcular_diaria(precipitacion_2021)
df_daily_4.head()

### Fuente 5: Precipitación_2026_1.csv

In [ ]:
precipitacion_2026 = pd.read_csv("Precipitación_2026_1.csv")
precipitacion_2026 = precipitacion_2026.rename(columns=RENAME_COLUMNAS)
precipitacion_2026["fecha"] = pd.to_datetime(
    precipitacion_2026["FechaObservacion"].str.slice(0, 10),
    errors="coerce",
    cache=True,
)

df_daily_5 = calcular_diaria(precipitacion_2026)
df_daily_5.head()

### Fuente 6: precipitacion_diaria.xlsx (Output4/Indicadores)

In [ ]:
PRECIP_DIARIA_PATH = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output4\Indicadores\precipitacion diaria\precipitacion_diaria.xlsx"

# Esta fuente ya viene agregada a nivel estacion-fecha (no requiere calcular_diaria)
precipitacion_diaria = pd.read_excel(PRECIP_DIARIA_PATH)
precipitacion_diaria = precipitacion_diaria.rename(columns=RENAME_COLUMNAS)
precipitacion_diaria["fecha"] = pd.to_datetime(precipitacion_diaria["fecha"], errors="coerce")

df_daily_6 = precipitacion_diaria[
    STATION_COLS + ["fecha", "precip_min_10min", "precip_max_10min",
                     "precip_media_10min", "precip_acum_diaria"]
].sort_values(["CodigoEstacion", "fecha"])

df_daily_6.head()

### Consolidado diario: concat de las 6 fuentes y verificación de duplicados

In [ ]:
df_daily_vf = pd.concat([df_daily, df_daily_2, df_daily_3, df_daily_4, df_daily_5, df_daily_6], ignore_index=True)

columnas_duplicado = ["fecha", "CodigoEstacion", "NombreEstacion", "Departamento", "Municipio"]
n_duplicados = df_daily_vf.duplicated(subset=columnas_duplicado).sum()
print(f"Filas duplicadas encontradas ({', '.join(columnas_duplicado)}): {n_duplicados}")

df_daily_vf = df_daily_vf.drop_duplicates(subset=columnas_duplicado)
print(df_daily_vf.shape)
df_daily_vf.head()

### Análisis exploratorio por estación (percentil 95 diario)

Alerta simple a nivel de estación (no forma parte del panel municipal que sigue más abajo).

In [81]:
df_daily_vf['fecha'] = pd.to_datetime(df_daily_vf['fecha'])

# Idempotente: si esta celda ya se corrio antes, elimina columnas previas
# antes de recalcular, para evitar colisiones de nombre en el merge
# (p95_x/p95_y) que rompen la linea siguiente con KeyError: 'p95'.
df_daily_vf = df_daily_vf.drop(columns=['mes', 'p95', 'extremo'], errors='ignore')

df_daily_vf['mes'] = df_daily_vf['fecha'].dt.month

# 1. Percentil 95 por estación y mes
p95 = df_daily_vf.groupby(['CodigoEstacion', 'mes'])['precip_acum_diaria'].quantile(0.95).reset_index()
p95.columns = ['CodigoEstacion', 'mes', 'p95']

# 2. Eventos extremos
df_daily_vf = df_daily_vf.merge(p95, on=['CodigoEstacion', 'mes'], how='left')
df_daily_vf['extremo'] = (df_daily_vf['precip_acum_diaria'] > df_daily_vf['p95']).astype(int)

# 3. Frecuencia y alerta por estación
estaciones_alerta = df_daily_vf.groupby(['CodigoEstacion', 'NombreEstacion', 'Departamento', 'Municipio', 
                                          'Latitud', 'Longitud'])['extremo'].mean().reset_index()

estaciones_alerta.columns = [*estaciones_alerta.columns[:-1], 'frecuencia_extremos']
estaciones_alerta['alerta'] = (estaciones_alerta['frecuencia_extremos'] > 0.1).astype(int)
print(estaciones_alerta['alerta'].value_counts())
estaciones_alerta.head()

alerta
0    2791
1     307
Name: count, dtype: int64


,CodigoEstacion,NombreEstacion,Departamento,Municipio,Latitud,Longitud,frecuencia_extremos,alerta
0,11017020,PR CHOCO: BAGADO,CHOCÓ,BAGADÓ,5.412000,-76.418000,0.060748,0
1,11025501,CARMEN DE ATRATO - AUT,CHOCO,EL CARMEN,5.888719,-76.145167,0.053353,0
2,11027030,EL SIETE,CHOCO,EL CARMEN,5.862000,-76.152056,0.003697,0
3,11027030,EL SIETE - AUT,CHOCO,EL CARMEN,5.862000,-76.152056,0.089400,0
4,11027070,BORAUDO,CHOCÓ,LLORÓ,5.515000,-76.576000,0.051806,0


## Panel hídrico municipal (exceso / déficit por municipio-mes)

Homologa cada estación a los municipios cercanos (radio máximo configurable) y construye el panel mensual municipio-año-mes con condición hídrica (Exceso / Déficit / Mixta / Normal).

### Configuración y parámetros

In [82]:
import numpy as np
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree

UMBRAL_DIA_HUMEDO_MM = 1.0   # WMO: dia de lluvia
PERCENTIL_EXCESO = 95
PERCENTIL_DEFICIT = 95
RADIO_MAX_KM = 100.0         # distancia maxima estacion-municipio para poder asignar dato

# --- Periodo de referencia fijo ---
# P95_lluvia, P95_racha y los umbrales mensuales de n_dias_exceso/n_dias_deficit
# se calibran SOLO con datos hasta esta fecha. Evita que el umbral se mueva
# cada vez que se agregan meses nuevos (lo que haria que meses ya clasificados
# cambien de condicion retroactivamente sin que haya pasado nada nuevo en ellos).
FECHA_CORTE_REFERENCIA = pd.Timestamp("2025-12-31")

# Percentil historico de dias-de-exceso/deficit por mes calendario que se
# considera "mes anomalo". Reemplaza la regla anterior de "al menos 1 dia".
PERCENTIL_MENSUAL_ALERTA = 90

# Minimo de dias con dato en el mes para que ese mes cuente en la
# calibracion del umbral y en la evaluacion (evita que meses con reporte
# muy parcial, ej. 2-3 dias, distorsionen la tasa exceso/deficit).
MIN_DIAS_VALIDOS_MES = 10

In [83]:
MENSUALES_DIR = r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Escenarios Cambio Climatico IDEAM IV comunicacion\Mensuales"


### Municipios (DIVIPOLA)

In [84]:
import geopandas as gpd

davipola = pd.read_excel(r"C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Escenarios Cambio Climatico IDEAM IV comunicacion\Mensuales\davipola_dane.xlsx")

gdf_mun = gpd.GeoDataFrame(
    davipola.rename(columns={"COD_MPIO": "codigo_dane", "NOM_MPIO": "nombre_municipio"}),
    geometry=gpd.points_from_xy(davipola["LONGITUD"], davipola["LATITUD"]),
    crs="EPSG:4326",
).to_crs("EPSG:3116")

gdf_mun.head()


,COD_DPTO,NOM_DPTO,codigo_dane,nombre_municipio,TIPO,LATITUD,LONGITUD,Geo Municipio,geometry
0,5,ANTIOQUIA,5001,MEDELLÍN,Municipio,6.257590,-75.611031,POINT (-75.61103107 6.257590259),POINT (830279.43 1183971.219)
1,5,ANTIOQUIA,5002,ABEJORRAL,Municipio,5.803728,-75.438474,POINT (-75.43847353 5.803728154),POINT (849255.441 1133713.635)
2,5,ANTIOQUIA,5004,ABRIAQUÍ,Municipio,6.627569,-76.085978,POINT (-76.08597756 6.627569378),POINT (777857.813 1225088.466)
3,5,ANTIOQUIA,5021,ALEJANDRÍA,Municipio,6.365534,-75.090597,POINT (-75.09059702 6.365534125),POINT (887908.38 1195770.751)
4,5,ANTIOQUIA,5030,AMAGÁ,Municipio,6.032922,-75.708003,POINT (-75.7080031 6.032921994),POINT (819468.682 1159148.246)


### Funciones: homologación estación-municipio, cálculo de exceso/déficit y agregación mensual

In [85]:
def construir_ranking_estacion_municipio(df_daily_vf: pd.DataFrame,
                                          gdf_mun: gpd.GeoDataFrame,
                                          crs_proyectado: str = "EPSG:3116",
                                          radio_max_km: float = RADIO_MAX_KM) -> pd.DataFrame:
    """
    Para cada municipio calcula el ranking de TODAS las estaciones dentro
    de radio_max_km, ordenadas de la mas cercana a la mas lejana. Esta
    tabla larga (codigo_dane, CodigoEstacion, distancia_km, rank_distancia)
    permite luego, mes a mes, usar la estacion mas cercana y hacer fallback
    a la siguiente mas cercana cuando la primera no tenga dato ese mes.
    Municipios sin ninguna estacion dentro del radio quedan sin filas.
    """
    estaciones = (
        df_daily_vf[["CodigoEstacion", "Latitud", "Longitud"]]
        .drop_duplicates(subset="CodigoEstacion")
        .reset_index(drop=True)
    )

    gdf_est = gpd.GeoDataFrame(
        estaciones,
        geometry=gpd.points_from_xy(estaciones["Longitud"], estaciones["Latitud"]),
        crs="EPSG:4326",
    ).to_crs(crs_proyectado)

    gdf_mun_proj = gdf_mun.to_crs(crs_proyectado).reset_index(drop=True)

    coords_est = np.column_stack([gdf_est.geometry.x, gdf_est.geometry.y])
    coords_mun = np.column_stack([gdf_mun_proj.geometry.x, gdf_mun_proj.geometry.y])

    tree = cKDTree(coords_est)
    vecinos_por_municipio = tree.query_ball_point(coords_mun, r=radio_max_km * 1000)

    filas = []
    for i, idx_estaciones in enumerate(vecinos_por_municipio):
        if not idx_estaciones:
            continue
        dist_m = np.linalg.norm(coords_est[idx_estaciones] - coords_mun[i], axis=1)
        for j, d in zip(idx_estaciones, dist_m):
            filas.append((
                gdf_mun_proj.at[i, "codigo_dane"],
                gdf_mun_proj.at[i, "nombre_municipio"],
                gdf_est.at[j, "CodigoEstacion"],
                d / 1000,
            ))

    ranking = pd.DataFrame(
        filas, columns=["codigo_dane", "nombre_municipio", "CodigoEstacion", "distancia_km"]
    ).sort_values(["codigo_dane", "distancia_km"])

    ranking["rank_distancia"] = ranking.groupby("codigo_dane").cumcount() + 1
    return ranking.reset_index(drop=True)


In [86]:
def calcular_exceso_hidrico(df_daily_vf: pd.DataFrame,
                             umbral_dia_humedo: float = UMBRAL_DIA_HUMEDO_MM,
                             percentil: int = PERCENTIL_EXCESO,
                             fecha_corte_referencia: pd.Timestamp = FECHA_CORTE_REFERENCIA) -> pd.DataFrame:
    """
    Marca flag_exceso=1 cuando precip_acum_diaria supera el percentil 95
    (sobre dias de lluvia >umbral) de la propia estacion.
    P95_lluvia se calibra SOLO con dias <= fecha_corte_referencia, y ese
    umbral fijo se aplica luego a TODA la serie (incluyendo fechas
    posteriores al corte), para que estas queden como evaluacion
    fuera de muestra y no contaminen su propio umbral.
    """
    df = df_daily_vf.copy()
    df["dia_lluvia"] = df["precip_acum_diaria"] > umbral_dia_humedo

    df_referencia = df[df["fecha"] <= fecha_corte_referencia]

    p95_lluvia = (
        df_referencia.loc[df_referencia["dia_lluvia"]]
        .groupby("CodigoEstacion")["precip_acum_diaria"]
        .quantile(percentil / 100)
        .rename("P95_lluvia")
    )

    df = df.merge(p95_lluvia, on="CodigoEstacion", how="left")
    df["flag_exceso"] = (df["precip_acum_diaria"] > df["P95_lluvia"]).astype(int)
    return df


In [87]:
def calcular_deficit_hidrico(df: pd.DataFrame,
                              umbral_dia_humedo: float = UMBRAL_DIA_HUMEDO_MM,
                              percentil: int = PERCENTIL_DEFICIT,
                              fecha_corte_referencia: pd.Timestamp = FECHA_CORTE_REFERENCIA) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calcula rachas de dias secos consecutivos por estacion (vectorizado) y
    marca flag_deficit=1 cuando la racha activa supera el percentil 95
    HISTORICO de esa estacion. Las rachas se identifican sobre la serie
    completa (una racha puede cruzar la fecha de corte), pero P95_racha se
    calibra usando solo las rachas que INICIARON antes o en la fecha de
    corte de referencia, para no dejar que rachas recientes definan su
    propio umbral de "normalidad".
    """
    df = df.sort_values(["CodigoEstacion", "fecha"]).reset_index(drop=True)
    df["dia_seco"] = df["precip_acum_diaria"] < umbral_dia_humedo

    cambio = (
        (df["dia_seco"] != df["dia_seco"].shift())
        | (df["CodigoEstacion"] != df["CodigoEstacion"].shift())
    )
    df["streak_id"] = cambio.cumsum()

    df["racha_actual"] = df.groupby("streak_id").cumcount() + 1
    df.loc[~df["dia_seco"], "racha_actual"] = 0

    rachas_hist = (
        df.loc[df["dia_seco"]]
        .groupby(["CodigoEstacion", "streak_id"])
        .agg(longitud_racha=("racha_actual", "max"), fecha_inicio=("fecha", "min"))
        .reset_index()
    )

    rachas_referencia = rachas_hist[rachas_hist["fecha_inicio"] <= fecha_corte_referencia]

    p95_racha = (
        rachas_referencia.groupby("CodigoEstacion")["longitud_racha"]
        .quantile(percentil / 100)
        .rename("P95_racha")
    )

    df = df.merge(p95_racha, on="CodigoEstacion", how="left")
    df["flag_deficit"] = (df["racha_actual"] > df["P95_racha"]).astype(int)

    return df, rachas_hist


In [88]:
def agregar_mensual_estacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega los flags diarios a nivel CodigoEstacion + anio + mes.
    n_dias_validos = dias con dato ese mes, para poder normalizar
    n_dias_exceso/n_dias_deficit y comparar meses de forma justa
    (distinta longitud de mes, reporte incompleto, etc.).
    """
    df = df.copy()
    df["anio"] = df["fecha"].dt.year
    df["mes"] = df["fecha"].dt.month

    agg = (
        df.groupby(["CodigoEstacion", "anio", "mes"])
        .agg(
            n_dias_exceso=("flag_exceso", "sum"),
            n_dias_deficit=("flag_deficit", "sum"),
            racha_maxima_mes=("racha_actual", "max"),
            n_dias_validos=("fecha", "count"),
        )
        .reset_index()
    )
    return agg


In [89]:
def calcular_umbral_mensual_historico(agg_estacion: pd.DataFrame,
                                       fecha_corte_referencia: pd.Timestamp = FECHA_CORTE_REFERENCIA,
                                       percentil: float = PERCENTIL_MENSUAL_ALERTA,
                                       min_dias_validos: int = MIN_DIAS_VALIDOS_MES) -> pd.DataFrame:
    """
    Umbral historico (por estacion + mes calendario) de que tan alta es
    "normalmente" la tasa de dias-de-exceso/deficit en ese mes. Se calcula
    SOLO con anios de referencia (<= fecha_corte_referencia) y solo con
    meses que tuvieron reporte suficiente (n_dias_validos >= min_dias_validos),
    para no medir el mes evaluado contra un umbral calibrado con el mismo
    mes, ni contaminar el umbral con meses de reporte muy parcial.
    Normaliza por n_dias_validos para comparar meses de distinta longitud
    o cobertura en igualdad de condiciones.
    """
    ref = agg_estacion[
        (agg_estacion["anio"] <= fecha_corte_referencia.year)
        & (agg_estacion["n_dias_validos"] >= min_dias_validos)
    ].copy()
    ref["tasa_exceso"] = ref["n_dias_exceso"] / ref["n_dias_validos"]
    ref["tasa_deficit"] = ref["n_dias_deficit"] / ref["n_dias_validos"]

    umbral = (
        ref.groupby(["CodigoEstacion", "mes"])
        .agg(
            umbral_tasa_exceso=("tasa_exceso", lambda s: s.quantile(percentil / 100)),
            umbral_tasa_deficit=("tasa_deficit", lambda s: s.quantile(percentil / 100)),
        )
        .reset_index()
    )
    return umbral


In [90]:
def agregar_mensual_municipio(agg_estacion: pd.DataFrame,
                               ranking_estacion_municipio: pd.DataFrame,
                               df_daily_vf: pd.DataFrame,
                               umbral_mensual: pd.DataFrame,
                               min_dias_validos: int = MIN_DIAS_VALIDOS_MES) -> pd.DataFrame:
    """
    Asigna a cada municipio-mes los valores de la estacion mas cercana
    (dentro de radio_max_km) que SI tenga dato ese mes. Si la estacion mas
    cercana no reporto ese anio-mes, se hace fallback a la siguiente mas
    cercana del ranking, y asi sucesivamente. Un municipio sin ninguna
    estacion dentro del radio, o cuyas estaciones cercanas no reportaron
    un mes dado, simplemente no obtiene fila para ese mes.

    alerta_exceso/alerta_deficit ya NO se activan con "al menos 1 dia":
    se activan solo si la tasa de dias afectados ese mes (n_dias / n_dias_validos)
    supera el umbral historico (percentil 90 por defecto) de esa estacion
    para ese mes calendario especifico. Meses con reporte muy parcial
    (n_dias_validos < min_dias_validos) no generan alerta (tasa poco confiable).
    """
    meta_est = (
        df_daily_vf[["CodigoEstacion", "Departamento", "ZonaHidrografica"]]
        .drop_duplicates(subset="CodigoEstacion")
    )

    candidatos = ranking_estacion_municipio.merge(agg_estacion, on="CodigoEstacion", how="inner")
    candidatos = candidatos.merge(meta_est, on="CodigoEstacion", how="left")
    candidatos = candidatos.merge(umbral_mensual, on=["CodigoEstacion", "mes"], how="left")

    candidatos = candidatos.sort_values(["codigo_dane", "anio", "mes", "rank_distancia"])
    panel = candidatos.drop_duplicates(subset=["codigo_dane", "anio", "mes"], keep="first").copy()

    panel = panel.rename(columns={
        "CodigoEstacion": "codigo_estacion_asignada",
        "distancia_km": "distancia_km_estacion_asignada",
    })

    panel["tasa_exceso"] = panel["n_dias_exceso"] / panel["n_dias_validos"]
    panel["tasa_deficit"] = panel["n_dias_deficit"] / panel["n_dias_validos"]

    reporte_suficiente = panel["n_dias_validos"] >= min_dias_validos

    panel["alerta_exceso"] = (
        reporte_suficiente & (panel["tasa_exceso"] > panel["umbral_tasa_exceso"])
    ).astype(int)
    panel["alerta_deficit"] = (
        reporte_suficiente & (panel["tasa_deficit"] > panel["umbral_tasa_deficit"])
    ).astype(int)

    return panel[
        ["codigo_dane", "nombre_municipio", "Departamento", "ZonaHidrografica",
         "anio", "mes", "codigo_estacion_asignada", "distancia_km_estacion_asignada",
         "n_dias_exceso", "n_dias_validos", "tasa_exceso", "alerta_exceso",
         "n_dias_deficit", "tasa_deficit", "alerta_deficit", "racha_maxima_mes"]
    ]


In [91]:
def clasificar_condicion_hidrica(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Clasifica cada fila municipio-mes en Mixta / Exceso / Deficit / Normal.
    """
    condiciones = [
        (panel["alerta_exceso"] == 1) & (panel["alerta_deficit"] == 1),
        (panel["alerta_exceso"] == 1) & (panel["alerta_deficit"] == 0),
        (panel["alerta_exceso"] == 0) & (panel["alerta_deficit"] == 1),
    ]
    etiquetas = ["Mixta", "Exceso", "Deficit"]
    panel["condicion_hidrica"] = np.select(condiciones, etiquetas, default="Normal")
    return panel


### Orquestador del pipeline

In [92]:
def construir_panel_hidrico_municipal(df_daily_vf: pd.DataFrame,
                                       gdf_mun: gpd.GeoDataFrame,
                                       mensuales_dir: str,
                                       radio_max_km: float = RADIO_MAX_KM) -> pd.DataFrame:
    """
    Ejecuta el pipeline completo y exporta df_panel_hidrico + tabla de
    auditoria de umbrales por estacion.
    """
    ranking_estacion_municipio = construir_ranking_estacion_municipio(
        df_daily_vf, gdf_mun, radio_max_km=radio_max_km
    )

    df_exceso = calcular_exceso_hidrico(df_daily_vf)
    df_full, rachas_hist = calcular_deficit_hidrico(df_exceso)

    agg_est = agregar_mensual_estacion(df_full)
    umbral_mensual = calcular_umbral_mensual_historico(agg_est)
    panel = agregar_mensual_municipio(agg_est, ranking_estacion_municipio, df_daily_vf, umbral_mensual)
    panel = clasificar_condicion_hidrica(panel)

    df_panel_hidrico = panel.rename(
        columns={"nombre_municipio": "municipio", "Departamento": "departamento",
                 "ZonaHidrografica": "zona_hidrografica"}
    )[
        ["codigo_dane", "municipio", "departamento", "zona_hidrografica",
         "anio", "mes", "codigo_estacion_asignada", "distancia_km_estacion_asignada",
         "n_dias_exceso", "n_dias_validos", "tasa_exceso", "alerta_exceso",
         "racha_maxima_mes", "n_dias_deficit", "tasa_deficit", "alerta_deficit",
         "condicion_hidrica"]
    ]

    print("Distribucion de condicion_hidrica:")
    print(df_panel_hidrico["condicion_hidrica"].value_counts())

    print("\nTop 10 municipios con mas meses en Deficit:")
    print(
        df_panel_hidrico.loc[df_panel_hidrico["condicion_hidrica"] == "Deficit"]
        .groupby("municipio").size().sort_values(ascending=False).head(10)
    )

    print("\nTop 10 municipios con mas meses en Exceso:")
    print(
        df_panel_hidrico.loc[df_panel_hidrico["condicion_hidrica"] == "Exceso"]
        .groupby("municipio").size().sort_values(ascending=False).head(10)
    )

    n_municipios_total = gdf_mun["codigo_dane"].nunique()
    n_municipios_con_dato = df_panel_hidrico["codigo_dane"].nunique()
    print(f"\nMunicipios con al menos un mes asignado (radio <= {radio_max_km} km): "
          f"{n_municipios_con_dato} / {n_municipios_total}")

    p95_lluvia = df_full.drop_duplicates("CodigoEstacion")[["CodigoEstacion", "P95_lluvia"]]
    p95_racha = df_full.drop_duplicates("CodigoEstacion")[["CodigoEstacion", "P95_racha"]]
    n_dias_lluvia = (
        df_full.groupby("CodigoEstacion")["dia_lluvia"].sum()
        .rename("n_dias_lluvia_historico").reset_index()
    )
    n_rachas = (
        rachas_hist.groupby("CodigoEstacion").size()
        .rename("n_rachas_historicas").reset_index()
    )

    auditoria_umbrales = (
        p95_lluvia.merge(p95_racha, on="CodigoEstacion")
        .merge(n_dias_lluvia, on="CodigoEstacion")
        .merge(n_rachas, on="CodigoEstacion", how="left")
    )
    auditoria_umbrales["n_rachas_historicas"] = auditoria_umbrales["n_rachas_historicas"].fillna(0).astype(int)

    out_panel = Path(mensuales_dir) / "panel_hidrico_municipal_percentiles.xlsx"
    out_audit = Path(mensuales_dir) / "auditoria_umbrales_estacion.xlsx"

    df_panel_hidrico.to_excel(out_panel, index=False)
    auditoria_umbrales.to_excel(out_audit, index=False)

    print(f"\nExportado: {out_panel}")
    print(f"Exportado: {out_audit}")

    return df_panel_hidrico


## Ejecución del pipeline y exportación

In [93]:
df_panel_hidrico = construir_panel_hidrico_municipal(df_daily_vf, gdf_mun, MENSUALES_DIR)


Distribucion de condicion_hidrica:
condicion_hidrica
Normal     192518
Deficit     23115
Exceso      20020
Mixta         745
Name: count, dtype: int64

Top 10 municipios con mas meses en Deficit:
municipio
BUENAVISTA    87
ALBANIA       80
LA UNIÓN      78
GRANADA       77
SAN PEDRO     70
EL PEÑÓN      67
VILLANUEVA    66
BOLÍVAR       66
CÓRDOBA       66
PALESTINA     65
dtype: int64

Top 10 municipios con mas meses en Exceso:
municipio
LA UNIÓN         81
SANTA BÁRBARA    74
BUENAVISTA       74
GUADALUPE        72
GRANADA          72
ARGELIA          68
PALESTINA        66
SAN FRANCISCO    65
FLORENCIA        63
SUCRE            62
dtype: int64

Municipios con al menos un mes asignado (radio <= 100.0 km): 1100 / 1121

Exportado: C:\Users\Usuario\OneDrive - Global Green Growth Institute\Documentos\2025\Outputs\Output1\Stress Test\3.Data\Escenarios Cambio Climatico IDEAM IV comunicacion\Mensuales\panel_hidrico_municipal_percentiles.xlsx
Exportado: C:\Users\Usuario\OneDrive - Global Gr

In [94]:
df_panel_hidrico.to_excel("datos_climaticos_panel.xlsx", index=False)